# 02 - Preprocessing

Cleans and imputes the Sentinel-2 satellite bands/indices produced by the EDA notebook. Missing satellite matches (cloud cover, no clear pass near the detection date) are flagged as a feature and imputed with the **regional** median rather than a global one.

In [1]:
# --- Notebook-chaining glue code (added when splitting the original pipeline) ---
import pandas as pd
import numpy as np

df = pd.read_csv("eda_output_combined.csv")
print(f"Loaded {len(df)} rows from eda_output_combined.csv")

Loaded 92869 rows from eda_output_combined.csv


## Step 2: Satellite (Sentinel-2) feature processing

Some rows have no usable satellite match (cloud cover, no clear pass near the
detection date) — this is a real, documented Sentinel-2 limitation, not a bug.
We flag it as its own feature, then impute missing values with the **regional**
median (not global) so a cloud-covered Texas point isn't imputed with an India-wide value.

In [2]:
SAT_BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
SAT_INDICES = ["ndvi", "ndbi", "nbr"]
SAT_COLS = SAT_BANDS + SAT_INDICES

df["satellite_data_available"] = df[SAT_COLS].notna().all(axis=1).astype(int)
missing_sat = (df["satellite_data_available"] == 0).sum()
print(f"Rows with no usable satellite match: {missing_sat} ({missing_sat/len(df)*100:.2f}%)")

for col in SAT_COLS:
    df[col] = df.groupby("region")[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())  # fallback

Rows with no usable satellite match: 487 (0.52%)


### Save output for the next notebook (feature engineering)

In [3]:
# --- Notebook-chaining glue code (added when splitting the original pipeline) ---
df.to_csv("preprocessing_output.csv", index=False)
print(f"Saved {len(df)} rows to preprocessing_output.csv for the feature engineering notebook.")

Saved 92869 rows to preprocessing_output.csv for the feature engineering notebook.
